In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "lonboard>=0.16.0",
#     "morecantile>=7.0.3",
#     "numpy>=2",
#     "httpx2",
#     "pillow>=12.1.1",
#     "sidecar>=0.8.1",
# ]
# ///

# NumPy Tiles from TiTiler in Lonboard

[TiTiler] is a dynamic tile server for raster data. Besides image formats like PNG, it can return a tile as a NumPy [`.npy`][npy] file, which holds the tile's pixel values as an array. This example requests tiles of a Sentinel-2 true-color image as `.npy` files from [titiler.xyz], TiTiler's public demo server, and renders them into images in Python with Lonboard's [`RasterLayer`].

![](../assets/raster-numpytiles.jpg)

It's the same image as in the [Visual Cloud-Optimized GeoTIFFs example], which reads the GeoTIFF from S3 itself. Here TiTiler reads the GeoTIFF, and our Python code only talks to TiTiler.

Note that this example uses `RasterLayer`'s generic constructor, whose arguments are private and experimental: they may change in a future release.

[TiTiler]: https://developmentseed.org/titiler/
[npy]: https://numpy.org/doc/stable/reference/generated/numpy.lib.format.html
[titiler.xyz]: https://titiler.xyz
[`RasterLayer`]: https://developmentseed.org/lonboard/latest/api/layers/raster-layer/
[Visual Cloud-Optimized GeoTIFFs example]: https://developmentseed.org/lonboard/latest/examples/raster-cog-rgb-server/

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run examples/raster-numpytiles.ipynb
```

(The `uvx` command is included when installing `uv`).

## Imports

First our imports. We use [httpx2] to make asynchronous HTTP requests to TiTiler, [morecantile] for the tile matrix set, and [pillow] (imported as `PIL`) for encoding image tiles to PNG.

[httpx2]: https://github.com/pydantic/httpx2
[morecantile]: https://developmentseed.org/morecantile/
[pillow]: https://pillow.readthedocs.io/en/stable/

In [ ]:
import io

import httpx2
import morecantile
import numpy as np
from morecantile.models import TMSBoundingBox
from PIL import Image
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard._geoarrow.ops import Bbox
from lonboard.raster import EncodedImage

## Get Dataset Metadata from TiTiler

TiTiler reads the Cloud-Optimized GeoTIFF for us, so all we need is its location, `src_path`. We pass it to TiTiler in the `url` query parameter of each request.

TiTiler's `/cog/{tileMatrixSetId}/tilejson.json` endpoint describes the dataset as a [TileJSON] document, including its bounds in longitude and latitude, its center as `[longitude, latitude, zoom]`, and its `minzoom` and `maxzoom` (8 and 14 for this scene). We'll use these to configure the layer.

[TileJSON]: https://github.com/mapbox/tilejson-spec

In [ ]:
src_path = "s3://sentinel-cogs/sentinel-s2-l2a-cogs/18/T/WL/2026/1/S2B_18TWL_20260101_0_L2A/TCI.tif"

In [ ]:
async def get_tilejson(src_path: str) -> dict:
    """Get tilejson docs."""
    async with httpx2.AsyncClient() as client:
        r = await client.get(
            "https://titiler.xyz/cog/WebMercatorQuad/tilejson.json",
            params={"url": src_path},
        )
        r.raise_for_status()

    return r.json()

In [ ]:
tilejson_docs = await get_tilejson(src_path)
bounds = tilejson_docs["bounds"]
center = tilejson_docs["center"]
minzoom = tilejson_docs["minzoom"]
maxzoom = tilejson_docs["maxzoom"]

## Create Fetch Callback

Lonboard's `RasterLayer` fetches tiles _through Python_. When the map needs a tile, it asks Python for it by its `x`, `y`, and `z` index. Python awaits our "fetch callback" to load the tile, passes the result to a "render callback" (next section), and transfers the rendered image to JavaScript for display.

Our fetch callback requests the tile from TiTiler's `/cog/tiles/{tileMatrixSetId}/{z}/{x}/{y}.npy` endpoint, asking for 256×256 pixels, the tile size of the `WebMercatorQuad` tile matrix set. Then it loads the response into an array with [`np.load`]. Two details matter here:

- **A new HTTP client for each request.** In JupyterLab, the tile callbacks run on a different asyncio event loop (in a kernel subshell thread) than the notebook's cells. An async HTTP client created or used in a cell must not be reused inside the fetch callback, so `titiler_fetch_tile` opens its own `httpx2.AsyncClient` for each tile.
- **A longer timeout.** titiler.xyz is a public demo server, and some tiles take longer than the client's default timeout of 5 seconds. A tile that fails isn't requested again, so we allow up to 30 seconds.

[`np.load`]: https://numpy.org/doc/stable/reference/generated/numpy.load.html

In [ ]:
async def titiler_fetch_tile(
    x: int,
    y: int,
    z: int,
) -> np.ndarray:
    """Fetch a specific tile from titiler."""
    # titiler.xyz is a shared demo server, and some tiles take longer than httpx's
    # 5-second default timeout. A tile that fails isn't requested again.
    async with httpx2.AsyncClient(timeout=30) as client:
        r = await client.get(
            f"https://titiler.xyz/cog/tiles/WebMercatorQuad/{z}/{x}/{y}.npy",
            params={"url": src_path, "tilesize": 256},
        )
        r.raise_for_status()

    return np.load(io.BytesIO(r.content))

## Create Render Callback

The render callback turns a fetched tile into an image. It must return an [`EncodedImage`], here a PNG. Since the rendering happens in Python, you can use _any Python code_ to shape how the tiles look.

For this 3-band `uint8` true-color image, each tile from TiTiler is a `(4, 256, 256)` `uint8` array: red, green, and blue, then a mask band that is 255 for valid pixels and 0 for nodata. The mask works as the alpha band as it is, making nodata pixels transparent. So `render_rgb_tile` only needs to move the band axis last, into the `(height, width, bands)` order that pillow expects, and encode the array as a PNG.

[`EncodedImage`]: https://developmentseed.org/lonboard/latest/api/raster/#lonboard.raster.EncodedImage

In [ ]:
def render_rgb_tile(tile: np.ndarray) -> EncodedImage:
    """Convert the array data to an RGBA PNG."""
    # titiler's .npy tiles are (bands, height, width): R, G, B, then a mask band
    # that's 255 for valid pixels and 0 for nodata, so it works as the alpha band.
    # Move the band axis last to become (height, width, 4)
    rgba = np.moveaxis(tile, 0, -1)

    # Encode as PNG
    img = Image.fromarray(rgba)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

## Create Tile Matrix Set

The layer also needs to know how the tiles are laid out: their tile matrix set and its coordinate reference system (CRS). The tiles we request from TiTiler follow the `WebMercatorQuad` tile matrix set (the `{tileMatrixSetId}` in the URLs above), which we load from [morecantile]. Its CRS is EPSG:3857, Web Mercator.

Lonboard renders the layer in JavaScript with [deck.gl-raster]. Version 0.8.1, the one Lonboard bundles at the time of writing, requires the tile matrix set's optional `boundingBox` and only loads tiles inside it. morecantile's `WebMercatorQuad` has no `boundingBox`, so we copy it with one set to the dataset's bounds, converted to EPSG:3857. Once Lonboard bundles a deck.gl-raster release that includes [developmentseed/deck.gl-raster#684], this step won't be needed: passing the dataset's bounds as `extent` (below) will be enough.

[morecantile]: https://developmentseed.org/morecantile/
[deck.gl-raster]: https://github.com/developmentseed/deck.gl-raster
[developmentseed/deck.gl-raster#684]: https://github.com/developmentseed/deck.gl-raster/pull/684

In [ ]:
# deck.gl-raster 0.8.1 requires the TileMatrixSet's optional `boundingBox`, and
# only loads tiles inside it, so set it to the dataset's bounds in the TMS's CRS.
# Not needed once lonboard uses a deck.gl-raster release that includes
# https://github.com/developmentseed/deck.gl-raster/pull/684 (`extent` is enough).
west, south, east, north = bounds
web_mercator = morecantile.tms.get("WebMercatorQuad")
tms = web_mercator.model_copy(
    update={
        "boundingBox": TMSBoundingBox(
            lowerLeft=web_mercator.xy(west, south),
            upperRight=web_mercator.xy(east, north),
            crs=web_mercator.crs,
        ),
    },
)

## Create the Layer

Now we can create the `RasterLayer`, passing the tile matrix set, its CRS, and our two callbacks. `min_zoom`, `max_zoom`, and `extent` give the layer the dataset's zoom range and bounds from the TileJSON. `_bounds` and `_center` set the map's initial view: Lonboard centers the map on `_center` and picks a zoom level based on the size of `_bounds`.

The arguments that start with an underscore (`_tile_matrix_set`, `_crs`, `_fetch_tile`, `_render_tile`, `_bounds`, and `_center`) are private and experimental: they may change in a future release. `Bbox` is private too, imported from `lonboard._geoarrow.ops`.

In [ ]:
layer = RasterLayer(
    _tile_matrix_set=tms,
    _crs="EPSG:3857",
    _fetch_tile=titiler_fetch_tile,
    _render_tile=render_rgb_tile,
    min_zoom=minzoom,
    max_zoom=maxzoom,
    extent=bounds,
    _bounds=Bbox(*bounds),
    # tilejson's center is [longitude, latitude, zoom]
    _center=center[:2],
)

Now we can create a map and display it just like any other layer

In [ ]:
# In JupyterLab, split the screen to render the map on the right
sidecar = Sidecar(anchor="split-right")

In [ ]:
# Create the map
m = Map(layer, height=800)

In [ ]:
# Render the map in the split screen
with sidecar:
    display(m)

## Debugging the render callback

We can debug our `render_rgb_tile` function by passing in a `tile` we fetch ourselves. Here we use morecantile to find the tile that contains the center of the scene at zoom level 12. Because `titiler_fetch_tile` opens a new HTTP client for each request, we can call it from a notebook cell, too:

In [ ]:
# Fetch the tile that contains the center of the scene at zoom level 12
center_tile = web_mercator.tile(center[0], center[1], 12)
tile = await titiler_fetch_tile(center_tile.x, center_tile.y, center_tile.z)
tile.shape, tile.dtype

In [ ]:
render_rgb_tile(tile)